In [1]:
# ============================================================
# PATHVQA WITH PROPER BAN (Bilinear Attention Networks)
# Following paper [20]: GRU + Faster R-CNN + BAN + Low-rank approx
# WITH CORRECT 50/30/20 IMAGE-LEVEL SPLIT
# ============================================================

!pip -q install datasets timm nltk rouge-score scikit-learn

# =========================
# IMPORTS
# =========================
import os, re, gc, math, time, json, random, string, hashlib, warnings
import numpy as np, pandas as pd, matplotlib.pyplot as plt

from io import BytesIO
from PIL import Image
from tqdm.auto import tqdm
from collections import Counter

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torchvision.transforms as T
import torchvision.transforms.functional as TF
from torchvision.models.detection import fasterrcnn_resnet50_fpn

from datasets import load_dataset
from sklearn.metrics import accuracy_score
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from rouge_score import rouge_scorer

import nltk
nltk.download("stopwords", quiet=True)
from nltk.corpus import stopwords

warnings.filterwarnings("ignore")

# =========================
# SEED / DEVICE
# =========================
SEED = 42
random.seed(SEED); np.random.seed(SEED)
torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)

# =========================
# CONFIG
# =========================
DATASET_NAME = "flaviagiammarino/path-vqa"

IMAGE_SIZE = 224
BATCH_SIZE = 4
NUM_WORKERS = 0
EPOCHS = 5
LR = 1e-4
WEIGHT_DECAY = 1e-4

MAX_Q_LEN = 30
WORD_VOCAB_SIZE = 5000

EMBED_DIM = 300
GRU_HIDDEN = 1024
DROPOUT = 0.5

# BAN specific parameters
NUM_GLIMPSES = 8
BAN_HIDDEN = 1024
LOW_RANK = 512

OUTPUT_DIR = "./outputs_ban_proper"
os.makedirs(OUTPUT_DIR, exist_ok=True)

STOP_WORDS = set(stopwords.words("english"))
SMOOTH_FN = SmoothingFunction().method1
ROUGE_SCORER = rouge_scorer.RougeScorer(["rouge1","rougeL"], use_stemmer=True)

# =========================
# TEXT NORMALIZATION
# =========================
def normalize(x, unk="<unk>"):
    x = str(x).lower().strip()
    x = x.translate(str.maketrans("", "", string.punctuation))
    x = re.sub(r"\s+", " ", x)
    toks = [t for t in x.split() if t not in STOP_WORDS]
    return " ".join(toks) if toks else unk

def tokenize(x):
    t = x.split()
    return t if t else ["<unk>"]

# =========================
# IMAGE HANDLING
# =========================
def image_to_pil(obj):
    if isinstance(obj, Image.Image):
        return obj.convert("RGB")
    if isinstance(obj, dict) and obj.get("bytes"):
        return Image.open(BytesIO(obj["bytes"])).convert("RGB")
    if isinstance(obj, dict) and obj.get("path"):
        return Image.open(obj["path"]).convert("RGB")
    return Image.new('RGB', (224, 224), 'black')

def stable_image_id(img_obj, fallback="unknown"):
    try:
        if isinstance(img_obj, dict):
            if img_obj.get("path", None) is not None:
                return str(img_obj["path"])
            if img_obj.get("bytes", None) is not None:
                return hashlib.md5(img_obj["bytes"]).hexdigest()
        if isinstance(img_obj, Image.Image):
            img_rgb = img_obj.convert("RGB")
            buf = BytesIO()
            img_rgb.save(buf, format="PNG")
            return hashlib.md5(buf.getvalue()).hexdigest()
    except Exception:
        return str(fallback)
    return str(fallback)

# =========================
# LOAD DATA
# =========================
print("Loading dataset...")
dataset = load_dataset(DATASET_NAME)

# Get column names
all_split_names = list(dataset.keys())
base_cols = dataset[all_split_names[0]].column_names

def find_col(cols, candidates):
    for c in candidates:
        if c in cols:
            return c
    return None

IMAGE_COL = find_col(base_cols, ["image", "img"])
QUESTION_COL = find_col(base_cols, ["question", "query"])
ANSWER_COL = find_col(base_cols, ["answer", "answers"])
IMAGE_ID_COL = find_col(base_cols, ["image_id", "img_id", "id"])

print(f"IMAGE_COL: {IMAGE_COL}, QUESTION_COL: {QUESTION_COL}, ANSWER_COL: {ANSWER_COL}")

# Build full dataframe with proper image IDs
rows = []
for s in dataset:
    print(f"Processing {s} split...")
    for i in tqdm(range(len(dataset[s])), desc=f"Loading {s}"):
        try:
            ex = dataset[s][i]
            img_obj = ex[IMAGE_COL]
            
            # Get stable image ID
            if IMAGE_ID_COL and IMAGE_ID_COL in ex:
                image_id = str(ex[IMAGE_ID_COL])
            else:
                image_id = stable_image_id(img_obj, fallback=f"{s}_{i}")
            
            rows.append({
                "split": s,
                "idx": i,
                "image_id": image_id,
                "q_raw": ex.get(QUESTION_COL, ""),
                "a_raw": ex.get(ANSWER_COL, ""),
                "q": normalize(ex.get(QUESTION_COL, "")),
                "a": normalize(ex.get(ANSWER_COL, ""))
            })
        except Exception as e:
            print(f"Error processing {s}_{i}: {e}")
            continue

df = pd.DataFrame(rows)
print(f"Total samples: {len(df)}")
print(f"Unique images: {df['image_id'].nunique()}")

# =========================
# EXACT SPLIT AS REQUESTED (50/30/20 with seeded random)
# =========================
unique_images = sorted(df["image_id"].unique().tolist())

rng = random.Random(SEED)   # IMPORTANT: same seeded RNG
rng.shuffle(unique_images)

n_images = len(unique_images)
n_train = int(0.5 * n_images)
n_val = int(0.3 * n_images)

train_imgs = set(unique_images[:n_train])
val_imgs   = set(unique_images[n_train:n_train + n_val])
test_imgs  = set(unique_images[n_train + n_val:])

train_df = df[df.image_id.isin(train_imgs)].reset_index(drop=True)
val_df   = df[df.image_id.isin(val_imgs)].reset_index(drop=True)
test_df  = df[df.image_id.isin(test_imgs)].reset_index(drop=True)

print("\n===== IMAGE-LEVEL SPLIT (MATCHED) =====")
print("Train images:", train_df["image_id"].nunique(), " Train QA:", len(train_df))
print("Val images  :", val_df["image_id"].nunique(),   " Val QA  :", len(val_df))
print("Test images :", test_df["image_id"].nunique(),  " Test QA :", len(test_df))

# =========================
# VOCABULARY BUILDING
# =========================
wc = Counter()
for t in train_df.q.tolist() + train_df.a.tolist():
    wc.update(tokenize(t))

itos = ["<pad>", "<unk>"] + [w for w, _ in wc.most_common(WORD_VOCAB_SIZE - 2)]
stoi = {w: i for i, w in enumerate(itos)}
PAD = stoi["<pad>"]
UNK = stoi["<unk>"]

print(f"Vocabulary size: {len(stoi)}")

def encode_q(x):
    t = tokenize(x)[:MAX_Q_LEN]
    ids = [stoi.get(w, UNK) for w in t]
    ids = ids + [PAD] * (MAX_Q_LEN - len(ids))
    return ids, min(len(t), MAX_Q_LEN)

# =========================
# ANSWER SPACE
# =========================
ans_list = sorted(set(train_df.a.tolist()))
if "<unk>" not in ans_list:
    ans_list.append("<unk>")
a2i = {a: i for i, a in enumerate(ans_list)}
i2a = {i: a for a, i in a2i.items()}

print(f"Answer vocabulary size: {len(ans_list)}")

# =========================
# DATASET CLASS
# =========================
tf = T.Compose([T.Resize((224, 224)), T.ToTensor()])

class VQADataset(Dataset):
    def __init__(self, df):
        self.df = df
        
    def __len__(self):
        return len(self.df)
    
    def __getitem__(self, i):
        r = self.df.iloc[i]
        ex = dataset[r.split][r.idx]
        img = tf(image_to_pil(ex.get(IMAGE_COL)))
        q, ql = encode_q(r.q)
        return {
            "img": img,
            "q": torch.tensor(q, dtype=torch.long),
            "ql": torch.tensor(ql, dtype=torch.long),
            "y": torch.tensor(a2i.get(r.a, a2i["<unk>"]), dtype=torch.long),
            "gt": r.a,
            "q_raw": r.q_raw,
            "image_id": r.image_id
        }

train_dl = DataLoader(VQADataset(train_df), batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
val_dl = DataLoader(VQADataset(val_df), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)
test_dl = DataLoader(VQADataset(test_df), batch_size=BATCH_SIZE, num_workers=NUM_WORKERS)

# =========================
# FASTER R-CNN FEATURE EXTRACTOR
# =========================
class FasterRCNNFeatureExtractor(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = fasterrcnn_resnet50_fpn(pretrained=True)
        self.backbone = self.model.backbone
        self.rpn = self.model.rpn
        
        for param in self.backbone.parameters():
            param.requires_grad = False
        
        self.out_channels = 256
        
    def forward(self, x):
        features = self.backbone(x)
        feature_map = list(features.values())[0]
        return feature_map

# =========================
# PROPER BAN IMPLEMENTATION
# =========================
class BilinearAttention(nn.Module):
    def __init__(self, v_dim, q_dim, num_glimpses, hidden_size, low_rank):
        super().__init__()
        self.num_glimpses = num_glimpses
        self.low_rank = low_rank
        
        self.U = nn.Linear(v_dim, low_rank, bias=False)
        self.V = nn.Linear(q_dim, low_rank, bias=False)
        
        self.h = nn.Parameter(torch.Tensor(low_rank, num_glimpses))
        nn.init.xavier_uniform_(self.h)
        
        self.proj = nn.Linear(v_dim * num_glimpses, hidden_size)
        self.dropout = nn.Dropout(0.5)
        
    def forward(self, v, q, return_attention=False):
        u = self.U(v)
        v_proj = self.V(q).unsqueeze(1)
        bilinear = u * v_proj
        attention = torch.softmax(bilinear @ self.h, dim=1)
        attended = torch.einsum('bng,bnd->bgd', attention, v)
        attended = attended.reshape(attended.size(0), -1)
        output = self.dropout(self.proj(attended))
        
        if return_attention:
            return output, attention
        return output

class GRUEncoder(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim, num_layers=1, dropout=0.5):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.gru = nn.GRU(embed_dim, hidden_dim, num_layers, 
                         batch_first=True, bidirectional=False, dropout=dropout)
        self.dropout = nn.Dropout(dropout)
        
    def forward(self, x, lengths):
        x = self.dropout(self.embedding(x))
        packed = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), 
                                                   batch_first=True, enforce_sorted=False)
        _, hidden = self.gru(packed)
        hidden = hidden[-1]
        return self.dropout(hidden)

class BANModel(nn.Module):
    def __init__(self, vocab_size, num_answers, embed_dim=300, gru_dim=1024, 
                 num_glimpses=8, ban_hidden=1024, low_rank=512):
        super().__init__()
        
        self.image_encoder = FasterRCNNFeatureExtractor()
        self.image_proj = nn.Linear(self.image_encoder.out_channels, ban_hidden)
        
        self.question_encoder = GRUEncoder(vocab_size, embed_dim, gru_dim, dropout=DROPOUT)
        self.question_proj = nn.Linear(gru_dim, ban_hidden)
        
        self.ban = BilinearAttention(
            v_dim=ban_hidden,
            q_dim=ban_hidden,
            num_glimpses=num_glimpses,
            hidden_size=ban_hidden,
            low_rank=low_rank
        )
        
        self.classifier = nn.Sequential(
            nn.Linear(ban_hidden, 1024),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(1024, num_answers)
        )
        
    def forward(self, images, questions, lengths, return_attention=False):
        v = self.image_encoder(images)
        B, C, H, W = v.shape
        v = v.view(B, C, H*W).permute(0, 2, 1)
        v = self.image_proj(v)
        
        q = self.question_encoder(questions, lengths)
        q = self.question_proj(q)
        
        if return_attention:
            fused, attention = self.ban(v, q, return_attention=True)
            att_map = attention.mean(dim=2).view(B, H, W)
            logits = self.classifier(fused)
            return logits, att_map
        else:
            fused = self.ban(v, q, return_attention=False)
            logits = self.classifier(fused)
            return logits

# =========================
# INITIALIZE MODEL
# =========================
model = BANModel(
    vocab_size=len(stoi),
    num_answers=len(a2i),
    embed_dim=EMBED_DIM,
    gru_dim=GRU_HIDDEN,
    num_glimpses=NUM_GLIMPSES,
    ban_hidden=BAN_HIDDEN,
    low_rank=LOW_RANK
).to(DEVICE)

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)

print(f"Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

# =========================
# METRICS FUNCTIONS
# =========================
def compute_bleu(pred, gold):
    pred_tokens = pred.split()
    gold_tokens = gold.split()
    try:
        bleu1 = sentence_bleu([gold_tokens], pred_tokens, weights=(1,0,0,0), smoothing_function=SMOOTH_FN) * 100
        bleu2 = sentence_bleu([gold_tokens], pred_tokens, weights=(0.5,0.5,0,0), smoothing_function=SMOOTH_FN) * 100
        bleu3 = sentence_bleu([gold_tokens], pred_tokens, weights=(1/3,1/3,1/3,0), smoothing_function=SMOOTH_FN) * 100
        return bleu1, bleu2, bleu3
    except:
        return 0, 0, 0

def compute_rouge(pred, gold):
    try:
        scores = ROUGE_SCORER.score(gold, pred)
        return scores['rouge1'].fmeasure * 100, scores['rougeL'].fmeasure * 100
    except:
        return 0, 0

def compute_f1(pred, gold):
    pred_tokens = set(pred.split())
    gold_tokens = set(gold.split())
    if not pred_tokens or not gold_tokens:
        return 0
    common = pred_tokens & gold_tokens
    precision = len(common) / len(pred_tokens)
    recall = len(common) / len(gold_tokens)
    return 2 * precision * recall / (precision + recall) * 100 if (precision + recall) > 0 else 0

# =========================
# TRAINING AND EVALUATION
# =========================
def train_epoch():
    model.train()
    total_loss = 0
    for batch in tqdm(train_dl, desc="Training"):
        img = batch["img"].to(DEVICE)
        q = batch["q"].to(DEVICE)
        ql = batch["ql"].to(DEVICE)
        y = batch["y"].to(DEVICE)
        
        logits = model(img, q, ql)
        loss = F.cross_entropy(logits, y)
        
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        total_loss += loss.item()
    return total_loss / len(train_dl)

@torch.no_grad()
def evaluate(dl, save_path=None):
    model.eval()
    golds, preds = [], []
    rows = []
    
    for batch in tqdm(dl, desc="Evaluating"):
        img = batch["img"].to(DEVICE)
        q = batch["q"].to(DEVICE)
        ql = batch["ql"].to(DEVICE)
        
        logits = model(img, q, ql)
        pred_ids = torch.argmax(logits, dim=1).cpu().tolist()
        
        for i, pid in enumerate(pred_ids):
            pred = i2a[pid]
            gold = batch["gt"][i]
            preds.append(pred)
            golds.append(gold)
            rows.append({
                "image_id": batch["image_id"][i],
                "question": batch["q_raw"][i],
                "gold": gold,
                "pred": pred,
                "correct": gold == pred
            })
    
    if save_path:
        pd.DataFrame(rows).to_csv(save_path, index=False)
    
    # Compute metrics
    yesno_idx = [i for i, g in enumerate(golds) if g in ['yes', 'no']]
    open_idx = [i for i, g in enumerate(golds) if g not in ['yes', 'no']]
    
    metrics = {
        'overall_exact_match_pct': np.mean([g == p for g, p in zip(golds, preds)]) * 100 if golds else 0,
        'yes_no_accuracy_pct': 0,
        'open_exact_match_pct': 0,
        'macro_f1_pct': 0,
        'bleu1_pct': 0,
        'bleu2_pct': 0,
        'bleu3_pct': 0,
        'rouge1_f_pct': 0,
        'rougeL_f_pct': 0,
        'total_samples': len(golds)
    }
    
    if yesno_idx:
        yesno_golds = [golds[i] for i in yesno_idx]
        yesno_preds = [preds[i] for i in yesno_idx]
        metrics['yes_no_accuracy_pct'] = accuracy_score(yesno_golds, yesno_preds) * 100
    
    if open_idx:
        open_golds = [golds[i] for i in open_idx]
        open_preds = [preds[i] for i in open_idx]
        
        metrics['open_exact_match_pct'] = np.mean([g == p for g, p in zip(open_golds, open_preds)]) * 100
        metrics['macro_f1_pct'] = np.mean([compute_f1(p, g) for g, p in zip(open_golds, open_preds)])
        
        bleu1, bleu2, bleu3 = zip(*[compute_bleu(p, g) for g, p in zip(open_golds, open_preds)])
        metrics['bleu1_pct'] = np.mean(bleu1)
        metrics['bleu2_pct'] = np.mean(bleu2)
        metrics['bleu3_pct'] = np.mean(bleu3)
        
        rouge1, rougeL = zip(*[compute_rouge(p, g) for g, p in zip(open_golds, open_preds)])
        metrics['rouge1_f_pct'] = np.mean(rouge1)
        metrics['rougeL_f_pct'] = np.mean(rougeL)
    
    return metrics, rows

# =========================
# MAIN TRAINING LOOP
# =========================
print("\n" + "="*60)
print("TRAINING BAN MODEL (Following Paper [20])")
print("="*60)

for epoch in range(EPOCHS):
    print(f"\nEpoch {epoch+1}/{EPOCHS}")
    train_loss = train_epoch()
    print(f"Train Loss: {train_loss:.4f}")
    
    val_metrics, _ = evaluate(val_dl, f"{OUTPUT_DIR}/val_epoch{epoch+1}.csv")
    print(f"Val Exact Match: {val_metrics['overall_exact_match_pct']:.2f}%")
    print(f"Val Yes/No Acc: {val_metrics['yes_no_accuracy_pct']:.2f}%")

# =========================
# FINAL TEST EVALUATION
# =========================
print("\n" + "="*60)
print("FINAL TEST RESULTS")
print("="*60)

test_metrics, test_rows = evaluate(test_dl, f"{OUTPUT_DIR}/test_predictions.csv")

# Print comprehensive results
print("\n" + "="*100)
print("TEST METRICS (Following BAN Paper [20])")
print("="*100)
print(f"{'Metric':<30} {'Score':<10}")
print("-"*40)
print(f"{'Overall Exact Match':<30} {test_metrics['overall_exact_match_pct']:>6.2f}%")
print(f"{'Yes/No Accuracy':<30} {test_metrics['yes_no_accuracy_pct']:>6.2f}%")
print(f"{'Open-ended Exact Match':<30} {test_metrics['open_exact_match_pct']:>6.2f}%")
print(f"{'Macro F1 Score':<30} {test_metrics['macro_f1_pct']:>6.2f}%")
print(f"{'BLEU-1':<30} {test_metrics['bleu1_pct']:>6.2f}%")
print(f"{'BLEU-2':<30} {test_metrics['bleu2_pct']:>6.2f}%")
print(f"{'BLEU-3':<30} {test_metrics['bleu3_pct']:>6.2f}%")
print(f"{'ROUGE-1':<30} {test_metrics['rouge1_f_pct']:>6.2f}%")
print(f"{'ROUGE-L':<30} {test_metrics['rougeL_f_pct']:>6.2f}%")

# Print 10 sample predictions
print("\n" + "="*60)
print("10 SAMPLE TEST PREDICTIONS")
print("="*60)

for i, row in enumerate(test_rows[:10]):
    print(f"\n{i+1}. Image ID: {row['image_id']}")
    print(f"   Question: {row['question'][:100]}...")
    print(f"   Ground Truth: {row['gold']}")
    print(f"   Prediction: {row['pred']}")
    print(f"   Status: {'✓ CORRECT' if row['correct'] else '✗ WRONG'}")

# Save all results
pd.DataFrame([test_metrics]).to_csv(f"{OUTPUT_DIR}/test_metrics.csv", index=False)
pd.DataFrame(test_rows).to_csv(f"{OUTPUT_DIR}/test_predictions.csv", index=False)

print(f"\n✅ Results saved to {OUTPUT_DIR}")
print(f"   - test_metrics.csv: All metrics")
print(f"   - test_predictions.csv: All predictions with image IDs")
print("\n✅ Experiment Complete!")

DEVICE: cuda
Loading dataset...


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00007-f2d0e9ef9f022d(…):   0%|          | 0.00/42.8M [00:00<?, ?B/s]

data/train-00001-of-00007-47d8e0220bf6c9(…):   0%|          | 0.00/81.0M [00:00<?, ?B/s]

data/train-00002-of-00007-7fb5037c4c5da7(…):   0%|          | 0.00/104M [00:00<?, ?B/s]

...

Train Loss: 3.0071


Evaluating:   0%|          | 0/2469 [00:00<?, ?it/s]

Val Exact Match: 45.49%
Val Yes/No Acc: 78.20%

FINAL TEST RESULTS


Evaluating:   0%|          | 0/1642 [00:00<?, ?it/s]


TEST METRICS (Following BAN Paper [20])
Metric                         Score     
----------------------------------------
Overall Exact Match             46.03%
Yes/No Accuracy                 78.03%
Open-ended Exact Match          34.26%
Macro F1 Score                  34.77%
BLEU-1                          34.63%
BLEU-2                          11.80%
BLEU-3                           7.85%
ROUGE-1                         34.83%
ROUGE-L                         34.83%

10 SAMPLE TEST PREDICTIONS

1. Image ID: b618b30e9a551778b9d58b64e73ad102
   Question: is normal palmar creases present?...
   Ground Truth: <unk>
   Prediction: <unk>
   Status: ✓ CORRECT

2. Image ID: b618b30e9a551778b9d58b64e73ad102
   Question: where is this from?...
   Ground Truth: gastrointestinal system
   Prediction: urinary
   Status: ✗ WRONG

3. Image ID: b618b30e9a551778b9d58b64e73ad102
   Question: what is present?...
   Ground Truth: gastrointestinal
   Prediction: liver
   Status: ✗ WRONG

4. Image ID: b